# RETRY — Fetch Ticket Details

Notebook limpio para correr el flujo controlado de **Fetch Ticket Details RETRY** en Python.

## 1. Imports y configuración base

In [1]:
import os
import time
import warnings
import psycopg2
import pandas as pd

from pathlib import Path
from datetime import date, timedelta
from itertools import product
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable*"
)

## 2. Variables de entorno y conexión a Postgres

In [2]:
# Ajusta el path si tu .env vive en otra carpeta.
load_dotenv(".env")

DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_PORT = os.getenv("DB_PORT", "5432")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    conn.execute(text("select 1"))

print("DB connection OK")

DB connection OK


## 3. Parámetros del proceso

In [3]:
# Fechas del análisis
from_date = date(2025, 6, 8)
to_date = from_date + timedelta(days=42)

# Control de alcance para demo
# Sube este valor para ampliar el sample.
AGENCY_SAMPLE_SIZE = 999999

# Cache local para evitar pegarle a DB repetidamente durante desarrollo
USE_CACHE = True

# Usamos CSV para evitar problemas de compatibilidad con parquet/pyarrow en notebooks.
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(exist_ok=True)

## 4. Agencias activas y canales

In [4]:
active_agencies_query = """
SELECT id
FROM travel_agency
WHERE eps_start_date IS NOT NULL
  AND eps_stop_date IS NULL
  AND airline_client_id IS NULL
ORDER BY id
"""

travel_agencies = pd.read_sql(active_agencies_query, engine)

sales_channels = [
    "SABRE", "ABACUS",
    "AMADEUS",
    "WORLDSPAN", "GALILEO",
    "_ARC",
    "FLGX", "NDC_SPARK",
    "NDC_LATAM",
    "NDC_AVIANCA",
    "WEBL", "EDIS"
]

agency_ids = travel_agencies["id"].head(AGENCY_SAMPLE_SIZE).tolist()

print(f"Active agencies found: {len(travel_agencies)}")
print(f"Agencies in current run: {len(agency_ids)}")
print(f"Sales channels in current run: {len(sales_channels)}")

Active agencies found: 109
Agencies in current run: 109
Sales channels in current run: 12


## 5. Cargar SQL base de FTD

In [5]:
with open("sql/FTD.sql", "r") as file:
    ftd_query = file.read()

print("FTD SQL template loaded")

FTD SQL template loaded


## 6. Condiciones para tickets sin coupons y coupons purgados

In [6]:
no_coupons_where = """
AND NOT EXISTS (
    SELECT 1
    FROM coupon c
    WHERE c.ticket_id = t.id
)
"""

purged_coupons_where = """
AND EXISTS (
    SELECT 1
    FROM coupon c
    WHERE c.ticket_id = t.id
      AND COALESCE(c.status, 'NULL') IN ('Z', 'UNKN', 'UNMT', 'NULL')
      AND COALESCE(c.origin, 'UNKN') NOT IN ('UNKN', 'ARNK')
      AND COALESCE(c.destination, 'UNKN') NOT IN ('UNKN', 'ARNK')
      AND c.departed_at IS NOT NULL
)
"""

## 7. Funciones reutilizables

In [7]:
def render_ftd_query(template, **kwargs):
    """Renderiza placeholders simples {key} y dobles {{key}} sin usar .format().

    Esto evita romper JSON internos del SQL que también usan llaves.
    """
    query = template

    for key, value in kwargs.items():
        # Primero placeholders dobles para evitar resultados tipo {1}
        query = query.replace(f"{{{{{key}}}}}", str(value))

        # Luego placeholders simples
        query = query.replace(f"{{{key}}}", str(value))

    return query


def get_db_connection():
    return psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT
    )


def cache_path_for_pair(agency_id, sales_channel, from_date, to_date):
    safe_channel = sales_channel.replace("/", "_")
    filename = f"ftd_{agency_id}_{safe_channel}_{from_date}_{to_date}.csv"
    return CACHE_DIR / filename


def fetch_ftd_for_agency_channel(
    ftd_query_template,
    travel_agency_id,
    sales_channel,
    from_date,
    to_date,
    use_cache=True
):
    cache_path = cache_path_for_pair(
        travel_agency_id,
        sales_channel,
        from_date,
        to_date
    )

    if use_cache and cache_path.exists():
        return pd.read_csv(cache_path, parse_dates=["issue_date", "refund_limit_date"])

    query_without_coupons = render_ftd_query(
        ftd_query_template,
        travel_agency_id=travel_agency_id,
        sales_channel=sales_channel,
        from_date=from_date,
        to_date=to_date,
        additional_where=no_coupons_where,
    )

    query_with_coupons = render_ftd_query(
        ftd_query_template,
        travel_agency_id=travel_agency_id,
        sales_channel=sales_channel,
        from_date=from_date,
        to_date=to_date,
        additional_where=purged_coupons_where,
    )

    conn = get_db_connection()

    try:
        df_without = pd.read_sql_query(query_without_coupons, conn)
        df_with = pd.read_sql_query(query_with_coupons, conn)
    finally:
        conn.close()

    results = (
        pd.concat([df_without, df_with], ignore_index=True)
        .drop_duplicates(subset=["id"])
    )

    if not results.empty:
        results["issue_date"] = pd.to_datetime(results["issue_date"]).dt.date
        results["refund_limit_date"] = pd.to_datetime(results["refund_limit_date"]).dt.date

    results.to_csv(cache_path, index=False)

    return results


def fetch_ftd_for_pairs(
    ftd_query_template,
    agency_ids,
    sales_channels,
    from_date,
    to_date,
    limit=None,
    sleep_seconds=0,
    use_cache=True
):
    pairs = list(product(agency_ids, sales_channels))

    if limit is not None:
        pairs = pairs[:limit]

    results = []

    for idx, (agency_id, sales_channel) in enumerate(pairs, start=1):
        print(f"[{idx}/{len(pairs)}] agency={agency_id} channel={sales_channel}")

        try:
            df = fetch_ftd_for_agency_channel(
                ftd_query_template=ftd_query_template,
                travel_agency_id=int(agency_id),
                sales_channel=sales_channel,
                from_date=from_date,
                to_date=to_date,
                use_cache=use_cache
            )

            print(f"  → rows: {len(df)}")

            if not df.empty:
                results.append(df)

        except Exception as e:
            print(f"  → ERROR: {type(e).__name__}: {e}")

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    if results:
        return (
            pd.concat(results, ignore_index=True)
            .drop_duplicates(subset=["id"])
        )

    return pd.DataFrame()


def chunk_list(values, chunk_size=50000):
    values = list(values)

    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def fetch_ndc_order_flags(ticket_ids, chunk_size=50000):
    ticket_ids = list(pd.Series(ticket_ids).dropna().unique())

    if not ticket_ids:
        return pd.DataFrame(
            columns=["id", "latam_api_success", "latam_portal_success"]
        )

    results = []
    conn = get_db_connection()

    try:
        for ids in chunk_list(ticket_ids, chunk_size):
            ids_sql = ",".join(str(int(x)) for x in ids)

            query = f"""
            SELECT
                not2.ticket_id AS id,
                (
                    no2.extra_params::json->>'airline_latam_ndc_api'
                    = 'success'
                ) AS latam_api_success,
                (
                    no2.extra_params::json->>'airline_latam_ndcportal_web'
                    = 'success'
                ) AS latam_portal_success
            FROM ndc_order_ticket not2
            JOIN ndc_order no2
              ON no2.id = not2.ndc_order_id
            WHERE not2.ticket_id IN ({ids_sql})
            """

            df = pd.read_sql_query(query, conn)

            if not df.empty:
                results.append(df)

    finally:
        conn.close()

    if not results:
        return pd.DataFrame(
            columns=["id", "latam_api_success", "latam_portal_success"]
        )

    flags = pd.concat(results, ignore_index=True)

    flags = (
        flags
        .groupby("id", as_index=False)
        .agg({
            "latam_api_success": "any",
            "latam_portal_success": "any"
        })
    )

    return flags

## 8. Ejecutar extracción FTD controlada

In [8]:
ftd_results = fetch_ftd_for_pairs(
    ftd_query_template=ftd_query,
    agency_ids=agency_ids,
    sales_channels=sales_channels,
    from_date=from_date,
    to_date=to_date,
    use_cache=USE_CACHE
)

print(f"FTD results: {ftd_results.shape}")
ftd_results.head()

[1/1308] agency=1 channel=SABRE
  → rows: 1
[2/1308] agency=1 channel=ABACUS
  → rows: 0
[3/1308] agency=1 channel=AMADEUS
  → rows: 11
[4/1308] agency=1 channel=WORLDSPAN
  → rows: 0
[5/1308] agency=1 channel=GALILEO
  → rows: 0
[6/1308] agency=1 channel=_ARC
  → rows: 0
[7/1308] agency=1 channel=FLGX
  → rows: 0
[8/1308] agency=1 channel=NDC_SPARK
  → rows: 0
[9/1308] agency=1 channel=NDC_LATAM
  → rows: 11883
[10/1308] agency=1 channel=NDC_AVIANCA
  → rows: 0
[11/1308] agency=1 channel=WEBL
  → rows: 0
[12/1308] agency=1 channel=EDIS
  → rows: 3
[13/1308] agency=2 channel=SABRE
  → rows: 3
[14/1308] agency=2 channel=ABACUS
  → rows: 0
[15/1308] agency=2 channel=AMADEUS
  → rows: 5
[16/1308] agency=2 channel=WORLDSPAN
  → rows: 0
[17/1308] agency=2 channel=GALILEO
  → rows: 0
[18/1308] agency=2 channel=_ARC
  → rows: 0
[19/1308] agency=2 channel=FLGX
  → rows: 0
[20/1308] agency=2 channel=NDC_SPARK
  → rows: 0
[21/1308] agency=2 channel=NDC_LATAM
  → rows: 0
[22/1308] agency=2 channe

,id,travel_agency_id,ticket_num,issue_date,refund_limit_date,pnr,iata_code,gds,source,status,fare_calculation,category,raw,extra_params,coupon_count
0,416566116,1,5443486775511,2025-06-18,NaT,NaN,91501745,SABRE,PARENT_SABRE,NaN,NaN,FTD OK,"{""extractor_class"":""com.aleron.gds.sabre.servi...",{},NaN
1,481092792,1,2305454267734,2025-07-07,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,CONNECTION,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '2', 'n...",NaN
2,481076630,1,755454295844,2025-07-17,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,ENGINEERING,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '3', 'n...",NaN
3,481023831,1,2305454216228,2025-06-13,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,CONNECTION,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '2', 'n...",NaN
4,481023932,1,755454216200,2025-06-13,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,ENGINEERING,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '5', 'n...",NaN


## 9. Enriquecer con flags NDC

In [9]:
ndc_order_flags = fetch_ndc_order_flags(ftd_results["id"].unique())

classified_candidates = ftd_results.merge(
    ndc_order_flags,
    on="id",
    how="left"
)

classified_candidates["latam_api_success"] = (
    classified_candidates["latam_api_success"]
    .fillna(False)
    .astype(bool)
)

classified_candidates["latam_portal_success"] = (
    classified_candidates["latam_portal_success"]
    .fillna(False)
    .astype(bool)
)

print(f"Classified candidates: {classified_candidates.shape}")
classified_candidates.head()

Classified candidates: (701686, 17)


,id,travel_agency_id,ticket_num,issue_date,refund_limit_date,pnr,iata_code,gds,source,status,fare_calculation,category,raw,extra_params,coupon_count,latam_api_success,latam_portal_success
0,416566116,1,5443486775511,2025-06-18,NaT,NaN,91501745,SABRE,PARENT_SABRE,NaN,NaN,FTD OK,"{""extractor_class"":""com.aleron.gds.sabre.servi...",{},NaN,False,False
1,481092792,1,2305454267734,2025-07-07,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,CONNECTION,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '2', 'n...",NaN,False,False
2,481076630,1,755454295844,2025-07-17,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,ENGINEERING,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '3', 'n...",NaN,False,False
3,481023831,1,2305454216228,2025-06-13,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,CONNECTION,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '2', 'n...",NaN,False,False
4,481023932,1,755454216200,2025-06-13,NaT,NaN,91501745,AMADEUS,BSP,NaN,NaN,ENGINEERING,"{""extractor_class"":""com.aleron.gds.sabre.servi...","{'agentCode': '9150174', 'checkDigit': '5', 'n...",NaN,False,False


## 10. Resumen de categorías

In [10]:
category_summary = (
    classified_candidates["category"]
    .value_counts(dropna=False)
    .reset_index()
)

category_summary.columns = ["category", "ticket_count"]

category_summary

,category,ticket_count
0,CONNECTION,475847
1,OTHER,105604
2,NOT_FOUND,85815
3,FTD OK,25148
4,ENGINEERING,6529
5,RETRY,2743


## 11. Separar outputs operativos

In [11]:
classified_outputs = {
    "retry": classified_candidates[classified_candidates["category"] == "RETRY"].copy(),
    "connection": classified_candidates[classified_candidates["category"] == "CONNECTION"].copy(),
    "engineering": classified_candidates[classified_candidates["category"] == "ENGINEERING"].copy(),
    "other": classified_candidates[classified_candidates["category"] == "OTHER"].copy(),
    "not_found": classified_candidates[classified_candidates["category"] == "NOT_FOUND"].copy(),
    "ftd_ok": classified_candidates[classified_candidates["category"] == "FTD OK"].copy(),
}

for name, df in classified_outputs.items():
    print(f"{name}: {len(df)} rows")

retry: 2743 rows
connection: 475847 rows
engineering: 6529 rows
other: 105604 rows
not_found: 85815 rows
ftd_ok: 25148 rows


## 12. Revisar casos RETRY

In [12]:
retry_tickets = classified_outputs["retry"]

retry_preview_columns = [
    "id",
    "travel_agency_id",
    "ticket_num",
    "issue_date",
    "gds",
    "source",
    "status",
    "category",
    "raw"
]

retry_tickets[retry_preview_columns].head(20)

,id,travel_agency_id,ticket_num,issue_date,gds,source,status,category,raw
21796,444290487,16,4622236287301,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21797,444290485,16,4622236285117,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21798,444290490,16,4622236289337,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21799,444290486,16,4622236286752,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21800,444290492,16,4622236290596,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21801,444290493,16,4622236290597,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21802,444290497,16,4622236292812,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21803,444290498,16,4622236302618,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21804,444290503,16,4622236341775,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."
21805,444290502,16,4622236341774,2025-06-09,EDIS,BSP,NaN,RETRY,"{""extractor_class"":""com.aleron.gds.sabre.servi..."


## 13. Extraer motivo simple de RETRY

In [13]:
RETRY_REASON_PATTERNS = {
    "INVALID_RESOURCE_USAGE": "InvalidResourceUsageException",
    "CONNECTION_FAILED": "ConnectionFailedException",
    "UNABLE_TO_OPEN_STREAM": "Unable to open stream",
    "TIMEOUT": "Timeout during waiting on",
    "CARRIER_NOT_RESPONDING": "CARRIER NOT RESPONDING",
    "CURRENT_STATUS_NOT_FOUND": "Current status: Not found",
    "ERROR_718": "Error 718",
    "ERROR_901": "Error 901",
    "ERROR_118": "Error 118",
    "ERROR_155": "Error 155",
    "ERROR_81": "Error 81",
    "UNEXPECTED_SYSTEM_ERROR": "Unexpected system error",
}


def extract_retry_reason(raw):
    raw_text = "" if pd.isna(raw) else str(raw)

    for reason, pattern in RETRY_REASON_PATTERNS.items():
        if pattern.lower() in raw_text.lower():
            return reason

    return "UNKNOWN_RETRY_REASON"


if not retry_tickets.empty:
    retry_tickets = retry_tickets.copy()
    retry_tickets["retry_reason"] = retry_tickets["raw"].apply(extract_retry_reason)

retry_tickets[[
    "id",
    "ticket_num",
    "travel_agency_id",
    "gds",
    "category",
    "retry_reason"
]].head(20)

,id,ticket_num,travel_agency_id,gds,category,retry_reason
21796,444290487,4622236287301,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21797,444290485,4622236285117,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21798,444290490,4622236289337,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21799,444290486,4622236286752,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21800,444290492,4622236290596,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21801,444290493,4622236290597,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21802,444290497,4622236292812,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21803,444290498,4622236302618,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21804,444290503,4622236341775,16,EDIS,RETRY,INVALID_RESOURCE_USAGE
21805,444290502,4622236341774,16,EDIS,RETRY,INVALID_RESOURCE_USAGE


## 15. Exportar resultados

In [15]:
# Actualizar retry output con retry_reason, si aplica.
classified_outputs["retry"] = retry_tickets

for name, df in classified_outputs.items():
    output_path = EXPORT_DIR / f"{name}_tickets.csv"
    df.to_csv(output_path, index=False)
    print(f"{name}: {len(df)} rows → {output_path}")

summary_path = EXPORT_DIR / "category_summary.csv"
category_summary.to_csv(summary_path, index=False)

print(f"summary → {summary_path}")

retry: 2743 rows → exports/retry_tickets.csv
connection: 475847 rows → exports/connection_tickets.csv
engineering: 6529 rows → exports/engineering_tickets.csv
other: 105604 rows → exports/other_tickets.csv
not_found: 85815 rows → exports/not_found_tickets.csv
ftd_ok: 25148 rows → exports/ftd_ok_tickets.csv
summary → exports/category_summary.csv


## 16. Encolar tickets que son RETRY de GDS tradicionales

In [16]:
DRY_RUN = True

GDS_CHANNELS = [
    "SABRE", "ABACUS",
    "AMADEUS",
    "WORLDSPAN", "GALILEO", "APOLLO",
    "_ARC",
    "FLGX"
]

In [21]:
retry_candidates = classified_candidates.copy()

required_columns = [
    "id",
    "travel_agency_id",
    "ticket_num",
    "gds",
    "category",
    "raw",
    "pnr"
]

missing_columns = [
    col for col in required_columns
    if col not in retry_candidates.columns
]

print("shape:", retry_candidates.shape)
print("missing_columns:", missing_columns)

shape: (701686, 17)
missing_columns: []


In [22]:
GDS_CHANNELS = [
    "SABRE", "ABACUS",
    "AMADEUS",
    "WORLDSPAN", "GALILEO", "APOLLO",
    "_ARC",
    "FLGX"
]

gds_retry = retry_candidates[
    retry_candidates["gds"].isin(GDS_CHANNELS)
    & (
        retry_candidates["raw"].isna()
        | (retry_candidates["category"] == "RETRY")
        | (
            (retry_candidates["gds"] == "_ARC")
            & retry_candidates["raw"].fillna("").str.contains(
                "fetchPnr not implemented in arc",
                regex=False
            )
            & retry_candidates["pnr"].notna()
            & (retry_candidates["pnr"] != "")
        )
    )
].copy()

print(gds_retry.shape)

gds_retry[
    ["id", "travel_agency_id", "ticket_num", "gds", "category", "pnr"]
].head()

(5416, 17)


,id,travel_agency_id,ticket_num,gds,category,pnr
23663,447071697,21,753518838793,AMADEUS,RETRY,NaN
43253,470876293,32,752110456261,_ARC,ENGINEERING,NNNXF
43254,470876295,32,752110456263,_ARC,ENGINEERING,LXNP6
43257,470876294,32,752110456262,_ARC,ENGINEERING,LXNP6
43259,470876291,32,752110456259,_ARC,ENGINEERING,HTYXW


In [23]:
GDS_CHANNELS = [
    "SABRE",
    "ABACUS",
    "AMADEUS",
    "WORLDSPAN",
    "GALILEO",
    "APOLLO",
    "_ARC",
    "FLGX"
]

gds_retry = retry_candidates[
    retry_candidates["gds"].isin(GDS_CHANNELS)
    & (
        (retry_candidates["category"] == "RETRY")
        | retry_candidates["raw"].isna()
    )
].copy()

print(gds_retry.shape)

gds_retry["category"].value_counts(dropna=False)

(5408, 17)


category
ENGINEERING    4895
RETRY           513
Name: count, dtype: int64

In [25]:
from pathlib import Path

EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(exist_ok=True)

gds_enqueue_text = "\n--\n".join(
    gds_retry.apply(
        lambda row: (
            f"-t\n{int(row['travel_agency_id'])}\n"
            f"fetch-ticket-details-job\n"
            f"--ticket-id\n{int(row['id'])}"
        ),
        axis=1
    )
)

gds_enqueue_path = EXPORT_DIR / "gds_retry_enqueue.txt"

with open(gds_enqueue_path, "w") as file:
    file.write(gds_enqueue_text)

print(f"Tickets a encolar: {len(gds_retry)}")
print(f"Archivo generado: {gds_enqueue_path}")

Tickets a encolar: 5408
Archivo generado: exports/gds_retry_enqueue.txt


In [28]:
import subprocess

DRY_RUN = False

command = [
    "keybase", "chat", "upload",
    "--title", "!job-enqueue",
    "--channel", "bot",
    "dealengine",
    str(gds_enqueue_path)
]

if DRY_RUN:
    print("[DRY RUN] No se envió nada.")
    print(" ".join(command))
else:
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr)


Attachment upload starting
Attachment upload progress 0% (0 of 304982 bytes uploaded)
Attachment upload progress 10% (32768 of 304982 bytes uploaded)
Attachment upload progress 100% (304982 of 304982 bytes uploaded)



## 14. RETRY pendientes por canal NDC

Mostramos los tickets clasificados como RETRY que pertenecen a canales distintos de GDS tradicional.
Estos flujos se analizarán y encolarán en una siguiente iteración.

In [34]:
GDS_CHANNELS = [
    "SABRE",
    "ABACUS",
    "AMADEUS",
    "WORLDSPAN",
    "GALILEO",
    "APOLLO",
    "_ARC",
    "FLGX"
]

non_gds_retry = classified_candidates[
    ~classified_candidates["gds"].isin(GDS_CHANNELS)
    & (classified_candidates["category"] == "RETRY")
].copy()

In [35]:
retry_by_channel = (
    non_gds_retry["gds"]
    .value_counts()
    .reset_index()
)

retry_by_channel.columns = ["channel", "retry_count"]

retry_by_channel["percentage"] = (
    retry_by_channel["retry_count"]
    / retry_by_channel["retry_count"].sum()
    * 100
).round(2)

retry_by_channel

,channel,retry_count,percentage
0,EDIS,2229,99.96
1,NDC_LATAM,1,0.04


In [36]:
retry_by_agency_channel = (
    non_gds_retry
    .groupby(
        [
            "travel_agency_id",
            "gds"
        ]
    )
    .size()
    .reset_index(name="retry_count")
    .sort_values(
        "retry_count",
        ascending=False
    )
)

retry_by_agency_channel.head(50)

,travel_agency_id,gds,retry_count
0,16,EDIS,853
5,27,EDIS,683
13,124,EDIS,280
4,25,EDIS,194
8,40,EDIS,99
9,46,EDIS,29
14,148,EDIS,25
10,48,EDIS,21
1,18,EDIS,16
2,19,EDIS,14


In [37]:
retry_by_agency_channel["channel_total"] = (
    retry_by_agency_channel
    .groupby("gds")["retry_count"]
    .transform("sum")
)

retry_by_agency_channel["channel_percentage"] = (
    retry_by_agency_channel["retry_count"]
    / retry_by_agency_channel["channel_total"]
    * 100
).round(2)

retry_by_agency_channel.sort_values(
    ["gds", "retry_count"],
    ascending=[True, False]
)

,travel_agency_id,gds,retry_count,channel_total,channel_percentage
0,16,EDIS,853,2229,38.27
5,27,EDIS,683,2229,30.64
13,124,EDIS,280,2229,12.56
4,25,EDIS,194,2229,8.70
8,40,EDIS,99,2229,4.44
9,46,EDIS,29,2229,1.30
14,148,EDIS,25,2229,1.12
10,48,EDIS,21,2229,0.94
1,18,EDIS,16,2229,0.72
2,19,EDIS,14,2229,0.63


# Análisis de errores no reintentables

Analizamos los tickets clasificados como CONNECTION, ENGINEERING y OTHER.

In [38]:
def read_sql_file(path):
    with open(path, "r") as file:
        return file.read()


def run_sql_query(query):
    conn = get_db_connection()

    try:
        df = pd.read_sql_query(query, conn)
    finally:
        conn.close()

    return df

In [39]:
connection_ticket_ids = (
    classified_candidates[classified_candidates["category"] == "CONNECTION"]["id"]
    .dropna()
    .astype(int)
    .unique()
)

engineering_ticket_ids = (
    classified_candidates[classified_candidates["category"] == "ENGINEERING"]["id"]
    .dropna()
    .astype(int)
    .unique()
)

other_tickets = classified_candidates[
    classified_candidates["category"] == "OTHER"
].copy()

print("CONNECTION:", len(connection_ticket_ids))
print("ENGINEERING:", len(engineering_ticket_ids))
print("OTHER:", len(other_tickets))

CONNECTION: 475847
ENGINEERING: 6529
OTHER: 105604


In [44]:
def render_ticket_ids_query(query_template, ticket_ids):
    ticket_ids = list(ticket_ids)

    if not ticket_ids:
        return None

    ticket_ids_sql = ",".join(str(int(ticket_id)) for ticket_id in ticket_ids)

    return (
        query_template
        .replace("{{ticket_ids}}", ticket_ids_sql)
        .replace("{ticket_ids}", ticket_ids_sql)
        .replace("<<id_list>>", ticket_ids_sql)
    )

In [45]:
connection_query_template = read_sql_file(
    "sql/Connection_analysis.sql"
)

connection_query = render_ticket_ids_query(
    connection_query_template,
    connection_ticket_ids
)

if connection_query is None:
    connection_errors = pd.DataFrame()
else:
    connection_errors = run_sql_query(connection_query)

print(connection_errors.shape)
connection_errors.head()

(475847, 3)


,id,travel_agency_id,category
0,444180620,67,GDS-IATA match missing on configurations
1,444180621,67,GDS-IATA match missing on configurations
2,444180622,67,GDS-IATA match missing on configurations
3,444180623,67,GDS-IATA match missing on configurations
4,444180624,67,GDS-IATA match missing on configurations


In [47]:
engineering_query_template = read_sql_file(
    "sql/Engineering_analysis.sql"
)

engineering_query = render_ticket_ids_query(
    engineering_query_template,
    engineering_ticket_ids
)

if engineering_query is None:
    engineering_errors = pd.DataFrame()
else:
    engineering_errors = run_sql_query(engineering_query)

print(engineering_errors.shape)
engineering_errors.head()

(6529, 2)


,id,category
0,444226429,Wrong length PNR
1,444226446,Wrong length PNR
2,444226468,Wrong length PNR
3,444226522,Wrong length PNR
4,444226561,Wrong length PNR


In [58]:
engineering_errors = engineering_errors.merge(
    classified_candidates[
        [
            "id",
            "travel_agency_id"
        ]
    ],
    on="id",
    how="left"
)

engineering_errors.head()

,id,category,travel_agency_id
0,444226429,Wrong length PNR,10
1,444226446,Wrong length PNR,10
2,444226468,Wrong length PNR,10
3,444226522,Wrong length PNR,10
4,444226561,Wrong length PNR,10


In [54]:
def summarize_errors_by_agency_and_type(
    df,
    error_type_col="category"
):
    if df.empty:
        return pd.DataFrame(
            columns=[
                "travel_agency_id",
                error_type_col,
                "ticket_count",
                "agency_total",
                "agency_percentage"
            ]
        )

    summary = (
        df
        .groupby(
            [
                "travel_agency_id",
                error_type_col
            ]
        )
        .size()
        .reset_index(name="ticket_count")
    )

    summary["agency_total"] = (
        summary
        .groupby("travel_agency_id")["ticket_count"]
        .transform("sum")
    )

    summary["agency_percentage"] = (
        summary["ticket_count"]
        / summary["agency_total"]
        * 100
    ).round(2)

    return (
        summary
        .sort_values(
            [
                "ticket_count",
                "agency_percentage"
            ],
            ascending=[False, False]
        )
        .reset_index(drop=True)
    )

In [55]:
connection_summary = summarize_errors_by_agency_and_type(
    connection_errors,
    error_type_col="category"
)

connection_summary.head(50)

,travel_agency_id,category,ticket_count,agency_total,agency_percentage
0,53,GDS-IATA match missing on configurations,151950,153679,98.87
1,162,GDS-IATA match missing on configurations,72391,74645,96.98
2,98,GDS-IATA match missing on configurations,54411,54507,99.82
3,93,GDS-IATA match missing on configurations,45765,45806,99.91
4,59,GDS-IATA match missing on configurations,18438,18441,99.98
5,127,GDS-IATA match missing on configurations,13682,13682,100.00
6,105,Not authorized,12323,15444,79.79
7,101,Not authorized,10067,10193,98.76
8,81,GDS-IATA match missing on configurations,8983,8988,99.94
9,137,GDS-IATA match missing on configurations,7926,8075,98.15


In [59]:
engineering_summary = summarize_errors_by_agency_and_type(
    engineering_errors,
    error_type_col="category"
)

engineering_summary.head(50)

,travel_agency_id,category,ticket_count,agency_total,agency_percentage
0,149,Missing raw log,1812,1945,93.16
1,48,OTHER,1503,1898,79.19
2,35,Missing raw log,733,887,82.64
3,72,Historial status not found,504,609,82.76
4,48,FTD OK,344,1898,18.12
5,41,Wrong length PNR,284,284,100.00
6,67,Wrong length PNR,219,219,100.00
7,149,OTHER,127,1945,6.53
8,101,Tax error,124,152,81.58
9,161,Wrong length PNR,114,114,100.00


In [53]:
other_summary = (
    other_tickets
    .groupby("travel_agency_id")
    .size()
    .reset_index(name="ticket_count")
    .sort_values("ticket_count", ascending=False)
)

other_summary["total_other"] = other_summary["ticket_count"].sum()

other_summary["percentage"] = (
    other_summary["ticket_count"]
    / other_summary["total_other"]
    * 100
).round(2)

other_summary.head(50)

,travel_agency_id,ticket_count,total_other,percentage
0,1,11886,105604,11.26
46,114,8648,105604,8.19
71,177,8257,105604,7.82
31,84,7846,105604,7.43
32,85,7312,105604,6.92
11,27,5120,105604,4.85
47,117,4986,105604,4.72
9,24,4648,105604,4.40
45,113,3497,105604,3.31
70,175,3057,105604,2.89


## Reportes en Slack

In [78]:
import os
import requests
from pathlib import Path

EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(exist_ok=True)

SLACK_WEBHOOK_URL = os.getenv("SLACK_WEBHOOK_URL")

In [83]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

SLACK_WEBHOOK_URL = os.getenv("SLACK_WEBHOOK_URL")

print("Webhook loaded:", SLACK_WEBHOOK_URL is not None)
print("Starts with:", SLACK_WEBHOOK_URL[:30] if SLACK_WEBHOOK_URL else None)

Webhook loaded: True
Starts with: https://hooks.slack.com/servic


In [71]:
def build_error_detail(df, error_group, category_col="category"):
    if df.empty:
        return pd.DataFrame(
            columns=[
                "ticket_id",
                "agency",
                "error_group",
                "error_category",
                "raw_error"
            ]
        )

    detail = df.copy()

    # Agregamos agencia/raw desde classified_candidates si no vienen en el df original
    enrichment_cols = [
        "id",
        "travel_agency_id",
        "raw"
    ]

    detail = detail.merge(
        classified_candidates[enrichment_cols],
        on="id",
        how="left",
        suffixes=("", "_from_classified")
    )

    # Resolver travel_agency_id si venía duplicado
    if "travel_agency_id_from_classified" in detail.columns:
        detail["travel_agency_id"] = detail["travel_agency_id"].fillna(
            detail["travel_agency_id_from_classified"]
        )

    # Resolver raw si venía duplicado
    if "raw_from_classified" in detail.columns:
        if "raw" in detail.columns:
            detail["raw"] = detail["raw"].fillna(
                detail["raw_from_classified"]
            )
        else:
            detail["raw"] = detail["raw_from_classified"]

    detail = detail.rename(columns={
        "id": "ticket_id",
        "travel_agency_id": "agency",
        category_col: "error_category",
        "raw": "raw_error"
    })

    detail["error_group"] = error_group

    return detail[
        [
            "ticket_id",
            "agency",
            "error_group",
            "error_category",
            "raw_error"
        ]
    ]

In [72]:
connection_detail = build_error_detail(
    connection_errors,
    error_group="CONNECTION",
    category_col="category"
)

engineering_detail = build_error_detail(
    engineering_errors,
    error_group="ENGINEERING",
    category_col="category"
)

other_detail = build_error_detail(
    other_tickets,
    error_group="OTHER",
    category_col="category"
)

error_detail = pd.concat(
    [
        connection_detail,
        engineering_detail,
        other_detail
    ],
    ignore_index=True
)

error_detail_path = EXPORT_DIR / "ftd_error_detail.csv"

error_detail.to_csv(error_detail_path, index=False)

print(f"CSV generado: {error_detail_path}")
print(error_detail.shape)

CSV generado: exports/ftd_error_detail.csv
(587980, 5)


In [73]:
def top_10_agencies(df, error_group):
    if df.empty:
        return pd.DataFrame(
            columns=[
                "travel_agency_id",
                "ticket_count",
                "error_group"
            ]
        )

    return (
        df
        .groupby("travel_agency_id")
        .size()
        .reset_index(name="ticket_count")
        .assign(error_group=error_group)
        .sort_values("ticket_count", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )


top_connection_agencies = top_10_agencies(
    connection_errors,
    "CONNECTION"
)

top_engineering_agencies = top_10_agencies(
    engineering_errors,
    "ENGINEERING"
)

top_other_agencies = top_10_agencies(
    other_tickets,
    "OTHER"
)

In [74]:
def format_top_10_for_slack(df, title):
    if df.empty:
        return f"*{title}*\nNo tickets found."

    lines = [f"*{title}*"]

    for idx, row in df.iterrows():
        lines.append(
            f"{idx + 1}. Agency `{int(row['travel_agency_id'])}` — "
            f"*{int(row['ticket_count']):,}* tickets"
        )

    return "\n".join(lines)

In [75]:
total_connection = len(connection_errors)
total_engineering = len(engineering_errors)
total_other = len(other_tickets)

slack_message = f"""
:rotating_light: *FTD Error Report*

*Totales*
• Connection: *{total_connection:,}* tickets
• Engineering: *{total_engineering:,}* tickets
• Other: *{total_other:,}* tickets

{format_top_10_for_slack(top_connection_agencies, "Top 10 agencias — Connection")}

{format_top_10_for_slack(top_engineering_agencies, "Top 10 agencias — Engineering")}

{format_top_10_for_slack(top_other_agencies, "Top 10 agencias — Other")}

:paperclip: CSV detallado generado localmente:
`{error_detail_path}`
"""

print(slack_message)


:rotating_light: *FTD Error Report*

*Totales*
• Connection: *475,847* tickets
• Engineering: *6,529* tickets
• Other: *105,604* tickets

*Top 10 agencias — Connection*
1. Agency `53` — *153,679* tickets
2. Agency `162` — *74,645* tickets
3. Agency `98` — *54,507* tickets
4. Agency `93` — *45,806* tickets
5. Agency `59` — *18,441* tickets
6. Agency `105` — *15,444* tickets
7. Agency `127` — *13,682* tickets
8. Agency `101` — *10,193* tickets
9. Agency `81` — *8,988* tickets
10. Agency `104` — *8,819* tickets

*Top 10 agencias — Engineering*
1. Agency `149` — *1,945* tickets
2. Agency `48` — *1,898* tickets
3. Agency `35` — *887* tickets
4. Agency `72` — *609* tickets
5. Agency `41` — *284* tickets
6. Agency `67` — *219* tickets
7. Agency `101` — *152* tickets
8. Agency `68` — *146* tickets
9. Agency `161` — *114* tickets
10. Agency `166` — *89* tickets

*Top 10 agencias — Other*
1. Agency `1` — *11,886* tickets
2. Agency `114` — *8,648* tickets
3. Agency `177` — *8,257* tickets
4. Age

In [84]:
if not SLACK_WEBHOOK_URL:
    raise ValueError("SLACK_WEBHOOK_URL no está configurado en el .env")

response = requests.post(
    SLACK_WEBHOOK_URL,
    json={"text": slack_message}
)

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: ok
